In [3]:
# Importo las librerías para conectarme y trabajar con dataframes
from sqlalchemy import create_engine
import pandas as pd
import pyodbc
from urllib.parse import quote_plus
import os
from dotenv import load_dotenv


# Cargo las variables de conexión desde el archivo .env (override por si el kernel ya tenía valores cacheados)
load_dotenv(override=True)

True

In [4]:
# Configuro la conexión a PostgreSQL de origen usando las variables del .env
usuario = os.getenv("PG_USUARIO")
password = os.getenv("PG_PASSWORD")
servidor = os.getenv("PG_SERVIDOR")
puerto = os.getenv("PG_PUERTO")
base_datos = os.getenv("PG_BASE_DATOS")

cadena = (
    f"postgresql+psycopg2://{usuario}:{password}@"
    f"{servidor}:{puerto}/{base_datos}"
)

engine_origen = create_engine(cadena)

try:
    with engine_origen.connect() as conexion:
        print("Conexión exitosa")
except Exception as error:
    print(type(error))
    print(error)

Conexión exitosa


In [5]:
# Extraigo los detalles de venta desde la base de datos origen
consulta_detalle_venta = "SELECT * FROM tienda.detalle_venta"
df_detalle_venta = pd.read_sql_query(consulta_detalle_venta, engine_origen)

In [6]:
# Configuro la conexión al stage en SQL Server usando las variables del .env
servidor_sqlserver = os.getenv("SQLSERVER_SERVIDOR")
usuario_sqlserver = os.getenv("SQLSERVER_USUARIO")
password_sqlserver = os.getenv("SQLSERVER_PASSWORD")
base_datos_stage = os.getenv("SQLSERVER_BASE_DATOS")
esquema_stage = os.getenv("SQLSERVER_ESQUEMA")
driver_sqlserver = os.getenv("SQLSERVER_DRIVER")

cadena_sqlserver = (
    f"mssql+pyodbc://{usuario_sqlserver}:{quote_plus(password_sqlserver)}@"
    f"{servidor_sqlserver}/{base_datos_stage}?driver={quote_plus(driver_sqlserver)}"
)

engine_stage = create_engine(cadena_sqlserver)
try:
    with engine_stage.connect() as conexion:
        print("Conexión exitosa")
except Exception as error:
    print(type(error))
    print(error)

Conexión exitosa


In [7]:
# Agrego la fecha y hora de carga al dataframe de detalles de venta
df_detalle_venta["fecha_carga"] = pd.Timestamp.now()

In [8]:
# Reviso una muestra de detalles de venta antes de cargarla
df_detalle_venta.head()

,detalle_venta_id,venta_id,producto_id,cantidad,precio_unitario,costo_unitario,descuento,subtotal,fecha_creacion,fecha_carga
0,1,1,55,1,442.90,309.60,22.15,420.75,2026-09-16 14:22:49.159577,2026-09-30 14:12:09.688631
1,2,2,24,1,375.95,211.70,56.39,319.56,2026-09-16 14:22:49.159577,2026-09-30 14:12:09.688631
2,3,2,20,1,309.75,197.65,0.00,309.75,2026-09-16 14:22:49.159577,2026-09-30 14:12:09.688631
3,4,3,58,3,250.00,140.00,0.00,750.00,2026-09-16 14:22:49.159577,2026-09-30 14:12:09.688631
4,5,3,39,1,86.40,60.30,8.64,77.76,2026-09-16 14:22:49.159577,2026-09-30 14:12:09.688631


In [9]:
# Cargo los detalles de venta en la tabla del stage
df_detalle_venta.to_sql(
    name="detalle_venta",
    con=engine_stage,
    schema=esquema_stage,
    if_exists="append",
    index=False
)

17

In [10]:
# Verifico el contenido de detalles de venta en el stage
consulta_stage = f"SELECT * FROM {esquema_stage}.detalle_venta"
df_detalle_venta_stage = pd.read_sql_query(consulta_stage, engine_stage)
df_detalle_venta_stage.head()

,detalle_venta_id,venta_id,producto_id,cantidad,precio_unitario,costo_unitario,descuento,subtotal,fecha_creacion,fecha_carga
0,1,1,55,1,442.90,309.60,22.15,420.75,2026-09-16 14:22:49.160,2026-09-30 14:12:09.690
1,2,2,24,1,375.95,211.70,56.39,319.56,2026-09-16 14:22:49.160,2026-09-30 14:12:09.690
2,3,2,20,1,309.75,197.65,0.00,309.75,2026-09-16 14:22:49.160,2026-09-30 14:12:09.690
3,4,3,58,3,250.00,140.00,0.00,750.00,2026-09-16 14:22:49.160,2026-09-30 14:12:09.690
4,5,3,39,1,86.40,60.30,8.64,77.76,2026-09-16 14:22:49.160,2026-09-30 14:12:09.690
